# Cardio HSIC results: symbolic probability of ischemia

Reads the artefacts written by `cardio_hsic.ipynb` -- it does **not** retrain anything.
Run the training notebook first.

The KAAM symbolic model gives a closed-form expression for each mechanism. Because
`ischemia` is a discrete node, `kan_model_mixed` trains it as a classifier, so its formula
is a **logit**: the probability is recovered as `P(ischemia) = sigma(formula)`.

Since a KAAM is additive in its inputs, that logit splits into one term per parent plus a
constant. That decomposition is what both plots below are built from.

In [ ]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "..")))

import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sympy as sp

from datasets.cardio import get_cardio_graph
from plotting.cardio_formula import get_delta
from utils.paths import get_experiment_paths

# Point this at another tree only if the training notebook wrote somewhere else.
OUTPUT_DIR = None
paths = get_experiment_paths('cardio', output_dir=OUTPUT_DIR)
data_dir, samples_dir = str(paths.data), str(paths.samples)

graph_cardio = get_cardio_graph()
factual_eval_d = pd.read_csv(os.path.join(data_dir, 'cardio_factual_eval_d.csv'))
with open(os.path.join(data_dir, 'cardio_node_info.pkl'), 'rb') as f:
    node_info = pickle.load(f)

TARGET = 'ischemia'
# Same ordering kan_model_mixed uses for a node's inputs, so formula symbols x_1..x_n line up.
x_cols = list(graph_cardio.predecessors(TARGET))

print(f'Evaluation set : {factual_eval_d.shape}')
print(f'Target         : {TARGET}')
print(f'Parents        : {x_cols}')
print(f'Parent types   : { {c: node_info[c]["type"] for c in x_cols} }')

## The symbolic mechanism for ischemia

In [ ]:
with open(os.path.join(samples_dir, 'cardio_formulas_kaam.pkl'), 'rb') as f:
    formulae = pickle.load(f)

# get_delta renames the generic x_1..x_n symbols to the actual parent names.
formula_logit, delta_frames = get_delta(factual_eval_d[x_cols], formulae[TARGET])
delta = delta_frames[0]
const = float(delta['const'].iloc[0])

used = sorted((str(s) for s in formula_logit.free_symbols), key=x_cols.index)
dropped = [c for c in x_cols if c not in used]

# get_delta only emits columns for parents that survive pruning; reinstate the rest as zeros
# so every parent has a (possibly flat) contribution to plot.
delta_terms = delta.drop(columns='const').reindex(columns=x_cols, fill_value=0.0)

print('logit P(ischemia) =')
print(f'  {sp.expand(formula_logit)}\n')
print('P(ischemia) =')
print(f'  sigma({sp.expand(formula_logit)})\n')

print(f'Parents in the formula : {used}')
print(f'Pruned away            : {dropped if dropped else "none"}')
print(f'Constant term          : {const:.4f}')

# The additive split must reproduce the formula exactly.
check = delta_terms.sum(axis=1) + const
direct = np.array([
    float(formula_logit.subs({sp.symbols(c): factual_eval_d[c].iloc[i] for c in used}))
    for i in range(25)
])
assert np.allclose(check.iloc[:25], direct, atol=1e-6), 'additive decomposition does not match the formula'
print('\nAdditive decomposition verified against the formula (first 25 patients).')

sp.Eq(sp.Symbol('P_ischemia'), sp.Function('sigma')(sp.expand(formula_logit)))

## Partial dependence of P(ischemia) on each parent

For parent `j` swept over its observed range, the other parents are held at their mean
contribution across the evaluation set:

`P(ischemia | x_j = v) = sigma(const + sum_{k != j} mean(delta_k) + delta_j(v))`

Continuous axes are shown in original clinical units; `diabetes` is binary, so it gets two
points rather than a curve.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def term_values(var, grid_values):
    """Evaluate this parent's additive contribution over a grid, in standardised units."""
    if var not in used:
        return np.zeros_like(grid_values, dtype=float)
    dependent_part = formula_logit.as_independent(sp.symbols(var), as_Add=True)[1]
    single = sp.lambdify(sp.symbols(var), dependent_part, 'numpy')
    return np.asarray(single(grid_values), dtype=float) * np.ones_like(grid_values, dtype=float)


def denorm(var, values):
    if node_info[var]['type'] != 'continuous':
        return values
    return values * node_info[var]['std'] + node_info[var]['mean']


mean_delta = delta_terms.mean(axis=0)
baseline = const + mean_delta.sum()

n_cols = 2
n_rows = int(np.ceil(len(x_cols) / n_cols))
fig, axs = plt.subplots(n_rows, n_cols, figsize=(9, 3.2 * n_rows), squeeze=False)

for i, var in enumerate(x_cols):
    ax = axs[i // n_cols][i % n_cols]
    observed = factual_eval_d[var].to_numpy()
    is_continuous = node_info[var]['type'] == 'continuous'

    if is_continuous:
        grid = np.linspace(observed.min(), observed.max(), 200)
    else:
        grid = np.array(sorted(np.unique(observed)), dtype=float)

    # Swap this parent's mean contribution for its value on the grid.
    logit = baseline - mean_delta[var] + term_values(var, grid)
    prob = sigmoid(logit)
    x_plot = denorm(var, grid)

    if is_continuous:
        ax.plot(x_plot, prob, color='tab:blue', lw=2)
    else:
        # Markers only: a binary parent has no intermediate values to interpolate through.
        ax.plot(x_plot, prob, 'o', color='tab:blue', ms=9)
        ax.set_xticks(x_plot)
        ax.set_xlim(x_plot.min() - 0.3, x_plot.max() + 0.3)

    ax.axhline(sigmoid(baseline), color='grey', ls='--', lw=1,
               label=f'population avg = {sigmoid(baseline):.3f}')
    ax.set_xlabel(f"{var}{' (original units)' if is_continuous else ''}")
    ax.set_ylabel('P(ischemia)')
    ax.set_ylim(0, 1)
    ax.set_title(f'{var}' + ('' if var in used else '  [pruned from the formula]'))
    ax.legend(fontsize=8)
    assert np.all((prob >= 0) & (prob <= 1)), f'{var}: probabilities left [0, 1]'

for j in range(len(x_cols), n_rows * n_cols):
    axs[j // n_cols][j % n_cols].axis('off')

fig.suptitle('Partial dependence of P(ischemia) on its parents (KAAM symbolic)')
fig.tight_layout()
fig.savefig(paths.figures / 'ischemia_pdp.pdf', bbox_inches='tight')
plt.show()
print(f"Saved -> {paths.figures / 'ischemia_pdp.pdf'}")

## Companion view: each parent's contribution to the logit

The same decomposition on the logit scale, where the KAAM is additive and the per-parent
curves are exactly the symbolic terms. Follows the layout of `notebooks/alejandro/cardio.ipynb`.

In [ ]:
fig, axs = plt.subplots(n_rows, n_cols, figsize=(9, 3.2 * n_rows), squeeze=False)
avg_line = float(mean_delta.mean())

for i, var in enumerate(x_cols):
    ax = axs[i // n_cols][i % n_cols]
    order = np.argsort(factual_eval_d[var].to_numpy())
    x_plot = denorm(var, factual_eval_d[var].to_numpy()[order])
    y_plot = delta_terms[var].to_numpy()[order]

    if node_info[var]['type'] == 'continuous':
        ax.plot(x_plot, y_plot, color='tab:cyan', lw=2)
    else:
        ax.plot(x_plot, y_plot, 'o', color='tab:cyan', ms=6)
        ax.set_xticks(sorted(np.unique(x_plot)))

    ax.axhline(avg_line, color='b', ls='--', lw=1, label='avg contribution')
    ax.set_xlabel(f"{var}{'' if node_info[var]['type'] == 'discrete' else ' (original units)'}")
    ax.set_ylabel(r'$\Delta$ logit(P(ischemia))')
    ax.set_title(var)
    ax.legend(fontsize=8)

for j in range(len(x_cols), n_rows * n_cols):
    axs[j // n_cols][j % n_cols].axis('off')

fig.suptitle('Per-parent contribution to logit P(ischemia) (KAAM symbolic)')
fig.tight_layout()
fig.savefig(paths.figures / 'ischemia_pdp_logit.pdf', bbox_inches='tight')
plt.show()
print(f"Saved -> {paths.figures / 'ischemia_pdp_logit.pdf'}")

print('\nMean contribution per parent (logit scale):')
print(mean_delta.round(4).to_string())
print(f'constant: {const:.4f}')
print(f'baseline P(ischemia) at mean contributions: {sigmoid(baseline):.4f}')
print(f'observed P(ischemia) in the evaluation set: {factual_eval_d[TARGET].mean():.4f}')